In [0]:
-- League - Shot Type Conversion
-- Which shot types turn into goals most often.
-- Change the values in params only; everything below reads from it.
WITH params AS (                              -- change settings here only
  SELECT
    CAST(NULL AS STRING) AS team,  -- team abbreviation; CAST(NULL AS STRING) = every team
    20252026 AS season,            -- e.g. 20242025; CAST(NULL AS INT) = every season
    2 AS game_type                 -- 2 = regular season, 3 = playoffs
),
shots AS (
  SELECT f.shot_type, f.type_desc_key = 'goal' AS is_goal
  FROM nhl.gold.fact_event f
  JOIN nhl.gold.dim_game g ON g.game_id = f.game_id
  JOIN nhl.gold.dim_team t ON t.team_id = f.event_owner_team_id
  CROSS JOIN params pr
  WHERE f.type_desc_key IN ('goal', 'shot-on-goal') AND f.shot_type IS NOT NULL
    AND COALESCE(f.period_type, '') <> 'SO'
    AND g.game_type = pr.game_type
    AND (pr.season IS NULL OR g.season = pr.season)
    AND (pr.team IS NULL OR t.abbrev = pr.team)
)
SELECT shot_type,
       COUNT(*)                                       AS shots,
       COUNT_IF(is_goal)                              AS goals,
       ROUND(100.0 * COUNT_IF(is_goal) / COUNT(*), 1) AS conversion_pct
FROM shots
GROUP BY shot_type
ORDER BY conversion_pct DESC;